In [2]:
# -*- coding: utf-8 -*-
"""
가맹점 위기 신호 예측 모델 구축
---------------------------------
이 스크립트는 다음 단계를 통해 가맹점의 위기(폐업, 휴업 등)를 예측하는
분류 모델을 구축하고 평가합니다.

1. 데이터 불러오기 및 전처리
2. 데이터 분할 및 정규화/스케일링
3. 예측 모델 구축 및 검증 (GridSearchCV 활용)
4. 모델 평가 및 비교
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ==============================================================================
# 1단계: 데이터 불러오기 및 전처리
# ==============================================================================
print("--- 1단계: 데이터 불러오기 및 전처리 시작 ---")

# 데이터 불러오기
try:
    df = pd.read_csv('/content/drive/MyDrive/GamjaNeverDie/Total_Data/Total_Data_v2.csv')
    print("'Total_Data_v2.csv' 파일을 성공적으로 불러왔습니다.")
    print("데이터 샘플 (상위 5개):")
    print(df.head())
    print("\n데이터 정보:")
    df.info()
except FileNotFoundError:
    print("오류: 'Total_Data_v2.csv' 파일을 찾을 수 없습니다. 스크립트와 동일한 디렉토리에 파일이 있는지 확인해주세요.")
    # 데모 실행을 위해 가상 데이터프레임 생성
    print("실행 데모를 위해 가상 데이터프레임을 생성합니다.")
    data = {
        '거래변동성': np.random.rand(1000) * 10,
        '순이익률': np.random.rand(1000) * 20 - 5,
        '평균고객연령': np.random.randint(20, 60, 1000),
        '동종업계순위': np.random.rand(1000),
        '위기점수': np.random.randint(0, 101, 1000),
        '업종': np.random.choice(['음식점', '서비스업', '소매업'], 1000)
    }
    df = pd.DataFrame(data)
    df.iloc[::10, 0] = np.nan # 결측치 삽입
    print("\n가상 데이터가 생성되었습니다.")


# --- 종속변수(Y) 설정 ---
# 분석 결과를 바탕으로 산정한 위기 예측 점수를 이용하여 최종 이진 분류 종속변수(Y)를 정의합니다.
# 예: '위기점수'가 50점 이하이면 '위기(1)', 50점 초과면 '안정(0)'으로 설정
# 실제 데이터에 맞게 기준 점수(threshold)와 컬럼명을 수정해야 합니다.

crisis_score_column = '위기점수' # [사용자 입력] 위기 점수 컬럼명
target_column = '위기여부' # [사용자 입력] 생성할 종속변수(Y) 컬럼명
threshold = 50 # [사용자 입력] 위기/안정 분류 기준 점수

if crisis_score_column in df.columns:
    df[target_column] = (df[crisis_score_column] <= threshold).astype(int)
    print(f"\n종속변수 '{target_column}' 생성 완료. (기준 점수: {threshold}점 이하)")
    print(df[target_column].value_counts(normalize=True))
else:
    print(f"경고: '{crisis_score_column}' 컬럼이 없어 종속변수를 생성할 수 없습니다. 가상 종속변수를 생성합니다.")
    df[target_column] = np.random.randint(0, 2, df.shape[0])


# --- 결측치(NaN) 및 이상치 처리 ---
# 숫자형 데이터의 결측치는 중앙값(median)으로 대체
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"'{col}' 컬럼의 결측치를 중앙값({median_val:.2f})으로 대체했습니다.")

# 이상치 처리는 도메인 지식에 따라 달라질 수 있습니다.
# 예: IQR(Interquartile Range)을 이용한 이상치 제거 또는 대체


# --- 범주형 변수 처리 (One-Hot Encoding) ---
# X 변수 목록에 범주형 변수가 포함된 경우, 숫자형으로 변환합니다.
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
if len(categorical_cols) > 0:
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    print("\n범주형 변수에 대해 One-Hot Encoding을 적용했습니다.")
    print("처리 후 데이터 컬럼:", df.columns.tolist())

print("\n--- 1단계: 전처리 완료 ---\n")


# ==============================================================================
# 2단계: 변수 선택, 데이터 분할 및 정규화/스케일링
# ==============================================================================
print("\n--- 2단계: 변수 선택, 데이터 분할 및 스케일링 시작 ---")

# --- 사용할 X 변수 목록 정의 ---
# [사용자 입력] 분석에 사용할 숫자형(Numeric) 변수 목록입니다.
# 실제 데이터 파일에 있는 컬럼명 중에서, 분석에 의미있다고 생각하는 변수들을 직접 선택하고 수정해주세요.
feature_columns = [
    '배달매출금액 비율',
    '동일 업종 내 매출 순위 비율',
    '동일 상권 내 매출 순위 비율_x',
    '동일 업종 내 해지 가맹점 비중_x',
    '신규 고객 비중',
    '전월대비 매출금액 감소율(%)',
    '3개월 연속 감소 여부',
    '6개월 하락추세 여부',
    '매출금액 구간_회복지수_6개월',
    '급감여부(-20% YoY)',
    '매출 안정성(변동성) CV(3개월)',
    '매출탄력도(6개월)',
    '고객분포_다양성지수',
    '재방문 고객 비중_y'
]

# 사용할 변수(X)와 타겟(y) 최종 선택
X = df[feature_columns].copy()
y = df[target_column]

# --- 결측치(NaN) 처리 ---
# 선택된 X 변수 내의 결측치는 중앙값(median)으로 대체
X.fillna(X.median(), inplace=True)
print("\n선택된 X 변수의 결측치 처리를 완료했습니다.")

# [참고] 범주형 변수를 사용하고 싶을 경우, 아래와 같이 One-Hot Encoding을 적용할 수 있습니다.
# categorical_features = ['상권', '업종'] # 예시
# X_categorical = pd.get_dummies(df[categorical_features], drop_first=True)
# X = pd.concat([X, X_categorical], axis=1)


# --- 데이터 분할 (학습용: 80%, 평가용: 20%) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\n데이터 분할 완료:")
print(f"학습용 데이터: {X_train.shape}, 평가용 데이터: {X_test.shape}")


# --- 데이터 정규화/스케일링 ---
# Min-Max Scaling (정규화)
min_max_scaler = MinMaxScaler()
X_train_norm = min_max_scaler.fit_transform(X_train)
X_test_norm = min_max_scaler.transform(X_test)
print("\nMin-Max Scaling (정규화) 적용 완료.")

# Standard Scaling (표준화)
std_scaler = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train)
X_test_std = std_scaler.transform(X_test)
print("Standard Scaling (표준화) 적용 완료.")


# --- Grid Search 준비 (하이퍼파라미터 최적화) ---
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [5, 10]
}
print("\nGrid Search를 위한 파라미터 그리드가 준비되었습니다. (탐색 범위를 줄여 속도 개선)")
print("--- 2단계 완료 ---\n")

# ==============================================================================
# 3단계: 예측 모델 구축 및 검증
# ==============================================================================
print("--- 3단계: 예측 모델 구축 및 검증 시작 ---")
print("Standard Scaling(표준화)된 데이터를 사용하여 모델을 학습합니다.")

# 모델 정의
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

# 학습된 모델과 최적 파라미터를 저장할 딕셔너리
trained_models = {}

# RandomForest에 GridSearchCV 적용
print("\nRandom Forest 모델에 GridSearchCV를 적용하여 최적 하이퍼파라미터를 탐색합니다...")
rf = RandomForestClassifier(random_state=42)
grid_search_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf,
                              cv=3, n_jobs=-1, verbose=2, scoring='recall') # 재현율(Recall)을 기준으로 최적화
grid_search_rf.fit(X_train_std, y_train)

best_rf = grid_search_rf.best_estimator_
trained_models["Random Forest"] = best_rf
print(f"\nRandom Forest 최적 파라미터: {grid_search_rf.best_params_}")


# 나머지 모델 학습
print("\n다른 모델들의 학습을 시작합니다...")
for name, model in models.items():
    if name != "Random Forest":
        model.fit(X_train_std, y_train)
        trained_models[name] = model
        print(f"{name} 모델 학습 완료.")

print("\n--- 3단계: 모델 구축 완료 ---\n")


# ==============================================================================
# 4단계: 모델 평가 및 비교
# ==============================================================================
print("--- 4단계: 모델 평가 및 비교 시작 ---")

results = []

for name, model in trained_models.items():
    # 예측
    y_pred = model.predict(X_test_std)
    y_pred_proba = model.predict_proba(X_test_std)[:, 1]

    # 성능 지표 계산
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "AUC-ROC": roc_auc
    })

# 결과 비교를 위해 DataFrame으로 변환
results_df = pd.DataFrame(results)
results_df.set_index('Model', inplace=True)

print("--- 모델별 성능 평가 결과 ---")
print(results_df)

# 재현율(Recall)이 가장 높은 모델 선택
best_model_by_recall = results_df['Recall'].idxmax()
best_recall_score = results_df['Recall'].max()

print("\n--- 최종 결론 ---")
print("가맹점의 '위기' 상태를 놓치지 않고 예측하는 것이 중요하므로, 재현율(Recall)을 핵심 지표로 고려합니다.")
print(f"평가 결과, '{best_model_by_recall}' 모델이 재현율 {best_recall_score:.4f}로 가장 높은 성능을 보였습니다.")
print("따라서, 이 모델을 위기 신호 예측에 가장 효과적인 모델로 제안합니다.")
print("\n--- 모든 단계 완료 ---")

--- 1단계: 데이터 불러오기 및 전처리 시작 ---
'Total_Data_v2.csv' 파일을 성공적으로 불러왔습니다.
데이터 샘플 (상위 5개):
   Unnamed: 0     가맹점구분번호        기준년월 가맹점 운영개월수 구간   매출금액 구간   매출건수 구간  \
0        8768  13D9737A1D  2023-01-01     3_25-50%  2_10-25%   1_10%이하   
1       72685  D89DEC9B29  2023-01-01     4_50-75%   1_10%이하   1_10%이하   
2       39069  E6BF54CE96  2023-01-01     5_75-90%  4_50-75%   1_10%이하   
3       39065  E6738F086C  2023-01-01     5_75-90%  5_75-90%  4_50-75%   
4       39063  E629D92144  2023-01-01     3_25-50%  2_10-25%  3_25-50%   

  유니크 고객 수 구간    객단가 구간   취소율 구간  배달매출금액 비율  ...  매출금액 구간_회복지수_6개월  \
0     1_10%이하  5_75-90%  5_상위5구간       15.4  ...               NaN   
1     1_10%이하  4_50-75%  3_상위3구간       22.7  ...               NaN   
2    2_10-25%  5_75-90%  5_상위5구간        9.6  ...               0.0   
3    4_50-75%  5_75-90%  1_상위1구간  -999999.9  ...               NaN   
4    3_25-50%  2_10-25%  1_상위1구간  -999999.9  ...               NaN   

   급감여부(-20% YoY)  매출 안정성(변동성) CV(3개월)  매출탄력도(6개월

/tmp/ipython-input-2438180306.py:77: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)



범주형 변수에 대해 One-Hot Encoding을 적용했습니다.
처리 후 데이터 컬럼: ['Unnamed: 0', '배달매출금액 비율', '동일 업종 매출금액 비율_x', '동일 업종 매출건수 비율', '동일 업종 내 매출 순위 비율', '동일 상권 내 매출 순위 비율_x', '동일 업종 내 해지 가맹점 비중_x', '동일 상권 내 해지 가맹점 비중', '남성 20대이하 고객 비중', '남성 30대 고객 비중', '남성 40대 고객 비중', '남성 50대 고객 비중', '남성 60대이상 고객 비중', '여성 20대이하 고객 비중', '여성 30대 고객 비중', '여성 40대 고객 비중', '여성 50대 고객 비중', '여성 60대이상 고객 비중', '재방문 고객 비중_x', '신규 고객 비중', '거주 이용 고객 비율', '직장 이용 고객 비율', '유동인구 이용 고객 비율', '개설일', '폐업일', '전월대비 매출금액 감소율(%)', '3개월 연속 감소 여부', '6개월 하락추세 여부', '매출금액 구간_회복지수_6개월', '급감여부(-20% YoY)', '매출 안정성(변동성) CV(3개월)', '매출탄력도(6개월)', '매출 안정성(변동성) 여부', '동일 상권 내 매출 순위 비율_y', '동일 업종 내 해지 가맹점 비중_y', '고객분포_다양성지수', '동일 업종 매출금액 비율_y', '재방문 고객 비중_y', '위기여부', '가맹점구분번호_002816BA73', '가맹점구분번호_003473B465', '가맹점구분번호_003AC99735', '가맹점구분번호_0041E4E5AE', '가맹점구분번호_0050D68B18', '가맹점구분번호_00646B6673', '가맹점구분번호_0074C4990A', '가맹점구분번호_007BE37BA2', '가맹점구분번호_00803E9174', '가맹점구분번호_00A2CF886A', '가맹점구분번호_00BC189C4B', '가맹점구분번호_00CEAAD71A', '가맹점구분번호_00DA1813DE', '가맹점구분번호_00F

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [08:13:47] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost 모델 학습 완료.

--- 3단계: 모델 구축 완료 ---

--- 4단계: 모델 평가 및 비교 시작 ---
--- 모델별 성능 평가 결과 ---
                     Accuracy  Precision    Recall  F1 Score   AUC-ROC
Model                                                                 
Random Forest        0.495727   0.498272  0.580018  0.536046  0.493624
Logistic Regression  0.499769   0.501535  0.657278  0.568941  0.498457
SVM                  0.501848   0.503082  0.666245  0.573280  0.500000
XGBoost              0.501501   0.503596  0.523339  0.513277  0.498515

--- 최종 결론 ---
가맹점의 '위기' 상태를 놓치지 않고 예측하는 것이 중요하므로, 재현율(Recall)을 핵심 지표로 고려합니다.
평가 결과, 'SVM' 모델이 재현율 0.6662로 가장 높은 성능을 보였습니다.
따라서, 이 모델을 위기 신호 예측에 가장 효과적인 모델로 제안합니다.

--- 모든 단계 완료 ---
